In [1]:
#final 
import cv2
import numpy as np
import tkinter as tk
from tkinter import filedialog, Label, Button, Canvas, Frame, ttk, Toplevel
from PIL import Image, ImageTk
import time
import winsound
import random
import os

class FinalIntelligenceSim:
    def __init__(self, root):
        self.root = root
        self.root.title("Sentinel-1 AI SAR Analysis - Final Round")
        self.root.geometry("1200x950") 
        self.root.configure(bg="#1e272e")
        self.current_spill_data = None 
        self.setup_ui()

    def setup_ui(self):
        # Header
        self.header_frame = Frame(self.root, bg="#1e272e")
        self.header_frame.pack(fill="x", pady=10)
        
        self.alert_light = Canvas(self.header_frame, width=30, height=30, bg="#1e272e", highlightthickness=0)
        self.alert_light.pack(side="left", padx=20)
        self.light_circle = self.alert_light.create_oval(5, 5, 25, 25, fill="#2f3640") # Initial Gray

        Label(self.header_frame, text="🛰️OIL SPILL DETECTION AND CONTROL",
              font=("Arial", 22, "bold"), bg="#1e272e", fg="#00cec9").pack(side="left")

        main_frame = Frame(self.root, bg="#1e272e")
        main_frame.pack(expand=True, fill="both", padx=15)

        # MAIN View Canvas (Using White Boundary for "Old Logic" Clean Look)
        self.canvas = Canvas(main_frame, width=700, height=450, bg="#2f3640", 
                             highlightthickness=3, highlightbackground="#00cec9")
        self.canvas.grid(row=0, column=0, padx=(0, 20), pady=10)

        # Sidebar
        sidebar = Frame(main_frame, bg="#2d3436", width=400, padx=20, pady=20)
        sidebar.grid(row=0, column=1, sticky="nsew")

        Label(sidebar, text="📊 CONTROL PANEL", font=("Arial", 14, "bold"), 
              bg="#2d3436", fg="white").pack(pady=10)
        
        self.status_lbl = Label(sidebar, text="SYSTEM READY", font=("Arial", 16, "bold"), bg="#2d3436", fg="#00ff00")
        self.status_lbl.pack(pady=5)

        # Buttons
        Button(sidebar, text="📂 Scan Sentinel-1 Image", command=self.run_mission, 
               bg="#0984e3", fg="white", font=("Arial", 12, "bold"), width=28).pack(pady=8)

        # The NEW Button to open the comparison diagnostic
        self.intel_btn = Button(sidebar, text="ℹ️ Advanced Diagnostics", 
                                 command=self.open_diagnostic_panel, state="disabled",
                                 bg="#8e44ad", fg="white", font=("Arial", 12, "bold"), width=28)
        self.intel_btn.pack(pady=8)

        # NEW Button for Live Drone Simulation
        self.tracker_btn = Button(sidebar, text="📡 Live Mission Tracker", command=self.open_live_simulation, state="disabled", bg="#d35400", fg="white", font=("Arial", 12, "bold"), width=28)
        self.tracker_btn.pack(pady=8)

        self.remedy_btn = Button(sidebar, text="🛠️ Quick Drone Deploy", command=lambda: self.log("Quick Deployment active. Bypassing simulation for emergency response..."), 
                                 state="disabled", bg="#27ae60", fg="white", font=("Arial", 12, "bold"), width=28)
        self.remedy_btn.pack(pady=8)

        # Legend Box
        self.legend_frame = Frame(sidebar, bg="#2d3436", pady=10)
        self.legend_frame.pack(fill="x")
        self.leg_leg = Label(self.legend_frame, text="ANALYSIS LEGEND:", font=("Arial", 10, "bold"), bg="#2d3436", fg="white")
        self.leg_leg.pack(anchor="w")
        self.leg_h = Label(self.legend_frame, text="⚪ White Boundary: Verified Hydrocarbon Boundary", fg="white", bg="#2d3436", font=("Arial", 10))
        self.leg_h.pack(anchor="w")
        self.leg_m2 = Label(self.legend_frame, text="🔴 Red: Heavy Crude Core", fg="#ff4757", bg="#2d3436", font=("Arial", 10, "bold"))
        self.leg_m2.pack(anchor="w")
        self.leg_e = Label(self.legend_frame, text="🟢 Green: Thin Slick / Emulsion", fg="#2ecc71", bg="#2d3436", font=("Arial", 10))
        self.leg_e.pack(anchor="w")

        # Log Box
        self.log_box = Label(self.root, text="System initialized. Monitoring Arabian Sea Sector...\n", 
                             font=("Courier", 11), bg="black", fg="#00ff00", height=10, anchor="nw", justify="left", padx=15)
        self.log_box.pack(side="bottom", fill="x")

    def log(self, msg):
        current = self.log_box.cget("text")
        new_text = f"> {msg}\n" + "\n".join(current.split("\n")[:8])
        self.log_box.config(text=new_text)
        self.root.update()

    def trigger_alarm(self):
        # 3 Beeps + Red Light Flash
        for _ in range(3):
            self.alert_light.itemconfig(self.light_circle, fill="#ff4757") # Red
            self.root.update()
            winsound.Beep(1200, 400)
            self.alert_light.itemconfig(self.light_circle, fill="#2f3640") # Off
            self.root.update()
            time.sleep(0.1)
        self.alert_light.itemconfig(self.light_circle, fill="#ff4757") # Keep Red

    def run_mission(self):
        path = filedialog.askopenfilename()
        if not path: return

        self.log(f"Processing SAR Dataset: {os.path.basename(path)}")
        img = cv2.imread(path)
        img_disp = cv2.resize(img, (700, 450))
        gray = cv2.cvtColor(img_disp, cv2.COLOR_BGR2GRAY)
        total_px = gray.shape[0] * gray.shape[1]

        # PRIMARY SCAN: Clean Logic (One color boundary)
        # IMPROVED ACCURACY: Lower threshold & smaller area filter to catch isolated patches
        _, thresh = cv2.threshold(gray, 60, 255, cv2.THRESH_BINARY_INV)
        cnts, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        
        # Draw White Boundary for Clean Look
        # Catching smaller fragments (area > 30 instead of 100)
        area_px = sum(cv2.contourArea(c) for c in cnts if cv2.contourArea(c) > 30)
        cv2.drawContours(img_disp, [c for c in cnts if cv2.contourArea(c) > 30], -1, (255, 255, 255), 2)

        # DUAL THRESHOLD (The NEW High-Accuracy Model)
        # 1. Heavy Oil (Darkest cores)
        _, heavy_mask = cv2.threshold(gray, 45, 255, cv2.THRESH_BINARY_INV)
        # 2. Emulsification / Thin Slick (Mid-tone greys)
        emul_mask = cv2.inRange(gray, 46, 95)
        
        heavy_cnts, _ = cv2.findContours(heavy_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        emul_cnts, _ = cv2.findContours(emul_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        h_px = sum(cv2.contourArea(c) for c in heavy_cnts if cv2.contourArea(c) > 30)
        e_px = sum(cv2.contourArea(c) for c in emul_cnts if cv2.contourArea(c) > 20)
        total_px = h_px + e_px
        
        # Geolocation & Simulation Data
        # Random start point in the Arabian Sea / Gulf area
        lat, lon = random.uniform(18.0, 25.0), random.uniform(60.0, 72.0)
        total_oil_m2 = (h_px + e_px) * 120 # 1px approx 10m on Sentinel-1
        
        self.current_spill_data = {
            "h_m2": h_px * 120, "e_m2": e_px * 120,
            "total_m2": total_oil_m2,
            "coverage": ((h_px + e_px) / total_px) * 100,
            "start_coords": (lat, lon),
            "payload": (total_oil_m2 / 1000) * 0.8, # Tons of dispersant
            "coords": f"{lat:.4f}°N, {lon:.4f}°E",
            "path": path
        }

        self.tk_img = ImageTk.PhotoImage(Image.fromarray(cv2.cvtColor(img_disp, cv2.COLOR_BGR2RGB)))
        self.canvas.create_image(0, 0, anchor="nw", image=self.tk_img)

        if area_px > 500:
            self.log(f"Anomalous Hydrocarbon footprint identified. Isolated patches identified. Coverage: {self.current_spill_data['coverage']:.2f}%")
            self.status_lbl.config(text="⚠ SPILL DETECTED", fg="#ff4757")
            self.log("Scan Complete. Initiating Alarm...")
            self.trigger_alarm()
            self.intel_btn.config(state="normal")
            self.tracker_btn.config(state="normal")
            self.remedy_btn.config(state="normal")
        else:
            self.status_lbl.config(text="CLEAR - NO SPILL", fg="#00ff00")
            self.alert_light.itemconfig(self.light_circle, fill="#2ecc71") # Green light for clean

    def open_diagnostic_panel(self):
        if not self.current_spill_data: return
        diag_win = Toplevel(self.root)
        diag_win.title("SAR Model Cross-Analysis Dashboard")
        diag_win.geometry("1100x800")
        diag_win.configure(bg="#2d3436")
        d = self.current_spill_data

        Label(diag_win, text="📊 ADVANCED DIAGNOSTIC PANELS", font=("Arial", 16, "bold"), bg="#2d3436", fg="#00cec9").pack(pady=15)
        
        panels_frame = Frame(diag_win, bg="#2d3436")
        panels_frame.pack(expand=True, fill="both", padx=20)

        # --- VIEW 1: THE OLD "CLEAN" MODEL ---
        view1_frame = Frame(panels_frame, bg="#1e272e", padx=10, pady=10, highlightthickness=1, highlightbackground="grey")
        view1_frame.grid(row=0, column=0, padx=10)
        Label(view1_frame, text="Map 1: General Boundaries (The 'Clean' Model)", font=("Arial", 11, "bold"), bg="#1e272e", fg="white").pack(pady=5)
        
        v1_canvas = Canvas(view1_frame, width=500, height=350, bg="black")
        v1_canvas.pack()
        
        # Reproducing the clean white boundaries from the past version
        img_v1 = cv2.imread(d['path'])
        img_v1 = cv2.resize(img_v1, (500, 350))
        gray_v1 = cv2.cvtColor(img_v1, cv2.COLOR_BGR2GRAY)
        _, thresh_v1 = cv2.threshold(gray_v1, 65, 255, cv2.THRESH_BINARY_INV) # Standard threshold
        cnts_v1, _ = cv2.findContours(thresh_v1, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(img_v1, [c for c in cnts_v1 if cv2.contourArea(c) > 30], -1, (255, 255, 255), 2) # White
        
        self.tk_v1 = ImageTk.PhotoImage(Image.fromarray(cv2.cvtColor(img_v1, cv2.COLOR_BGR2RGB)))
        v1_canvas.create_image(0, 0, anchor="nw", image=self.tk_v1)

        # --- VIEW 2: THE NEW HIGH-ACCURACY DUAL-ZONE MODEL ---
        view2_frame = Frame(panels_frame, bg="#1e272e", padx=10, pady=10, highlightthickness=1, highlightbackground="grey")
        view2_frame.grid(row=0, column=1, padx=10)
        Label(view2_frame, text="Map 2: Tactical Multi-Zone Analysis (Heavy vs Emulsion)", font=("Arial", 11, "bold"), bg="#1e272e", fg="white").pack(pady=5)
        
        v2_canvas = Canvas(view2_frame, width=500, height=350, bg="black")
        v2_canvas.pack()
        
        # Reproducing the detailed dual-color boundaries from the new version
        img_v2 = cv2.imread(d['path'])
        img_v2 = cv2.resize(img_v2, (500, 350))
        gray_v2 = cv2.cvtColor(img_v2, cv2.COLOR_BGR2GRAY)
        _, heavy_m2 = cv2.threshold(gray_v2, 40, 255, cv2.THRESH_BINARY_INV)
        emul_m2 = cv2.inRange(gray_v2, 41, 95)
        h_cnts2, _ = cv2.findContours(heavy_m2, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        e_cnts2, _ = cv2.findContours(emul_m2, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        
        cv2.drawContours(img_v2, [c for c in h_cnts2 if cv2.contourArea(c) > 30], -1, (0, 0, 255), 2)  # RED Core
        cv2.drawContours(img_v2, [c for c in e_cnts2 if cv2.contourArea(c) > 20], -1, (0, 255, 0), 1)  # GREEN Fringe
        
        self.tk_v2 = ImageTk.PhotoImage(Image.fromarray(cv2.cvtColor(img_v2, cv2.COLOR_BGR2RGB)))
        v2_canvas.create_image(0, 0, anchor="nw", image=self.tk_v2)

        # INTEGRATED DATA & LEGEND
        intel_frame = Frame(diag_win, bg="#1e272e", padx=20, pady=15)
        intel_frame.pack(fill="x", padx=20, pady=20)
        
        left_p = Frame(intel_frame, bg="#1e272e")
        left_p.pack(side="left", fill="both")
        
        h_tons = (d['h_m2']/1000) * 0.9 # 900kg polymer per km2
        e_tons = (d['e_m2']/1000) * 0.15 # 150kg dispersant per km2
        
        info = f"GPS Target: {d['coords']}\n" \
               f"⚪ Verified Hydrocarbon Boundary(White): {d['h_m2']+d['e_m2']:.2f} m²\n" \
               f"🔴 Heavy Core(Red): {d['h_m2']:.2f} m² | Recommended: Polymer Solidifier ({h_tons:.1f} Metric Tons)\n" \
               f"🟢 Emulsion Fringe(Green): {d['e_m2']:.2f} m² | Recommended: Surface Dispersant ({e_tons:.1f} Metric Tons)"
        
        Label(left_p, text="LIVE DATA & REMEDIATION PLAN", font=("Arial", 11, "bold"), bg="#1e272e", fg="#00ff00").pack(anchor="w")
        Label(left_p, text=info, font=("Courier", 11), bg="black", fg="#00ff00", justify="left").pack(anchor="w", pady=5)

    def open_live_simulation(self):
        if not self.current_spill_data: return
        sim_win = Toplevel(self.root)
        sim_win.title("Live Mission Command & Control")
        sim_win.geometry("1100x650")
        sim_win.configure(bg="#1e272e")

        d = self.current_spill_data
        
        # --- TOP HEADER ---
        header = Frame(sim_win, bg="#2d3436", pady=10)
        header.pack(fill="x")
        Label(header, text="📡 ARABIAN SEA SECTOR: LIVE DRONE DEPLOYMENT", 
              font=("Arial", 16, "bold"), bg="#2d3436", fg="#e67e22").pack()

        main_sim_frame = Frame(sim_win, bg="#1e272e")
        main_sim_frame.pack(expand=True, fill="both", padx=20, pady=20)

        # --- LEFT: DYNAMIC GRID MAP ---
        map_frame = Frame(main_sim_frame, bg="#1e272e")
        map_frame.grid(row=0, column=0, padx=10)
        
        Label(map_frame, text="TACTICAL COORDINATE GRID", font=("Arial", 10, "bold"), bg="#1e272e", fg="white").pack()
        self.sim_canvas = Canvas(map_frame, width=500, height=400, bg="#001a1a", highlightthickness=2, highlightbackground="#00cec9")
        self.sim_canvas.pack()

        # Draw Grid Lines for the military aesthetic
        for i in range(0, 500, 50):
            self.sim_canvas.create_line(i, 0, i, 400, fill="#003333")
            self.sim_canvas.create_line(0, i, 500, i, fill="#003333")

        # --- RIGHT: TELEMETRY & PAYLOAD ---
        telemetry_frame = Frame(main_sim_frame, bg="#2d3436", padx=20, pady=20, width=450)
        telemetry_frame.grid(row=0, column=1, sticky="nsew", padx=10)

        Label(telemetry_frame, text="REAL-TIME TELEMETRY", font=("Arial", 12, "bold"), bg="#2d3436", fg="#00ff00").pack(anchor="w", pady=5)
        
        # Live Stats
        self.sim_gps = Label(telemetry_frame, text=f"GPS: {d['coords']}", font=("Courier", 13), bg="#2d3436", fg="#00ff00")
        self.sim_gps.pack(anchor="w", pady=2)
        
        self.sim_status = Label(telemetry_frame, text="STATUS: STANDBY", font=("Courier", 13), bg="#2d3436", fg="white")
        self.sim_status.pack(anchor="w", pady=2)

        # Payload Interaction
        Label(telemetry_frame, text="\nDISPERSANT PAYLOAD CAPACITY", font=("Arial", 11, "bold"), bg="#2d3436", fg="white").pack(anchor="w")
        self.payload_bar = ttk.Progressbar(telemetry_frame, length=350, mode='determinate')
        self.payload_bar.pack(pady=5)
        self.payload_bar['value'] = 100 # Start Full
        
        self.payload_text = Label(telemetry_frame, text=f"AVAILABLE: {d['payload']:.2f} Tons", font=("Courier", 12), bg="#2d3436", fg="#00cec9")
        self.payload_text.pack(anchor="w")

        # Drone Emoji on Map
        self.drone_icon = self.sim_canvas.create_text(50, 350, text="🛸", font=("Arial", 24))

        def start_mission():
            path_points = [
                (100, 300, "NORTH-WEST"), (200, 150, "NORTH"), 
                (350, 100, "NORTH-EAST"), (450, 250, "EAST"), (250, 350, "RE-CENTERING")
            ]
            
            curr_lat, curr_lon = d['start_coords']
            total_payload = d['payload']
            
            for dest_x, dest_y, direction in path_points:
                self.sim_status.config(text=f"STATUS: MOVING {direction}", fg="#f1c40f")
                
                start_x, start_y = self.sim_canvas.coords(self.drone_icon)
                
                # Smooth movement steps
                steps = 40
                dx = (dest_x - start_x) / steps
                dy = (dest_y - start_y) / steps
                
                for s in range(steps):
                    self.sim_canvas.move(self.drone_icon, dx, dy)
                    
                    # Update Coordinates & Payload
                    curr_lat += random.uniform(-0.0005, 0.0005)
                    curr_lon += random.uniform(-0.0005, 0.0005)
                    self.payload_bar['value'] -= 0.4
                    current_tonnage = total_payload * (self.payload_bar['value'] / 100)
                    
                    # Leave a trail on map
                    x, y = self.sim_canvas.coords(self.drone_icon)
                    self.sim_canvas.create_oval(x, y, x+2, y+2, fill="#00ff00", outline="")

                    # Update UI in real-time
                    self.sim_gps.config(text=f"GPS: {curr_lat:.6f}°N, {curr_lon:.6f}°E")
                    self.payload_text.config(text=f"AVAILABLE: {max(0, current_tonnage):.2f} Tons")
                    
                    sim_win.update()
                    time.sleep(0.04)
                
                # Notification sound at each sector point
                winsound.Beep(800, 100)

            self.sim_status.config(text="STATUS: MISSION SUCCESS - AREA NEUTRALIZED", fg="#00ff00")
            self.log("Remediation Complete: Drone returning to carrier vessel.")

        btn_frame = Frame(sim_win, bg="#1e272e")
        btn_frame.pack(pady=10)
        Button(btn_frame, text="🚀 INITIATE FLIGHT SEQUENCE", command=start_mission, 
               bg="#27ae60", fg="white", font=("Arial", 12, "bold"), padx=20).pack()

if __name__ == "__main__":
    root = tk.Tk()
    app = FinalIntelligenceSim(root)
    root.mainloop()